## visualize segmentation results

In [1]:
# import libraries
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.transforms.functional import to_pil_image
from torch.utils.data import DataLoader
import yaml
import os
from PIL import Image
import tqdm

import datasets
import models
import utils

In [ ]:
# Polyp
# set data path
input_path = 'load/Kvasir-SEG-test/images'
gt_path = 'load/Kvasir-SEG-test/masks'

# check extension
valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

input_imgs = [f for f in os.listdir(input_path) if f.lower().endswith(valid_exts)]
input_imgs.sort()
gt_imgs = [f for f in os.listdir(gt_path) if f.lower().endswith(valid_exts)]
gt_imgs.sort()

In [2]:
# CAMO
# set data path
input_path = 'load/CAMO/Images/Test'
gt_path = 'load/CAMO/Test_gt'

# check extension
valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

input_imgs = [f for f in os.listdir(input_path) if f.lower().endswith(valid_exts)]
input_imgs.sort()
gt_imgs = [f for f in os.listdir(gt_path) if f.lower().endswith(valid_exts)]
gt_imgs.sort()

In [3]:

# set config 
config_path = 'save/train_CAMO_4626_0618/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [4]:
# set device/model
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

best_model_path = 'save/train_CAMO_4626_0618/model_epoch_best.pth'
sam_checkpoint = torch.load(best_model_path, map_location='cuda:0')

# set sam
model = models.make(config['model']).to(device)
model.load_state_dict(sam_checkpoint, strict=True)
model.eval()

RuntimeError: Error(s) in loading state_dict for SAM:
	Missing key(s) in state_dict: "image_encoder.prompt_generator.lightweight_mlp_0.0.weight", "image_encoder.prompt_generator.lightweight_mlp_0.0.bias", "image_encoder.prompt_generator.lightweight_mlp_5.0.weight", "image_encoder.prompt_generator.lightweight_mlp_5.0.bias", "image_encoder.prompt_generator.lightweight_mlp_6.0.weight", "image_encoder.prompt_generator.lightweight_mlp_6.0.bias", "image_encoder.prompt_generator.lightweight_mlp_8.0.weight", "image_encoder.prompt_generator.lightweight_mlp_8.0.bias", "image_encoder.prompt_generator.lightweight_mlp_9.0.weight", "image_encoder.prompt_generator.lightweight_mlp_9.0.bias", "image_encoder.prompt_generator.lightweight_mlp_11.0.weight", "image_encoder.prompt_generator.lightweight_mlp_11.0.bias". 

In [16]:
# dataloader 

spec = config['test_dataset']

dataset = datasets.make(spec['dataset'])
dataset = datasets.make(spec['wrapper'], args = {'dataset': dataset})

loader = DataLoader(dataset, batch_size = spec['batch_size'], num_workers=8)


/home/yn-jang/anaconda3/envs/aas/lib/python3.8/site-packages/torchvision/transforms/transforms.py:329: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [17]:

def visualize_batch_sample(inp, gt, pred, save_path=None):
    """
    inp: Tensor (B, 3, H, W)
    gt: Tensor (B, 1, H, W) or (B, H, W\)
    pred: Tensor (B, 1, H, W) or (B, H, W)
    idx: 배치 내 이미지 index
    """

    # 배치에서 idx번째 이미지 선택
    # image = to_pil_image(inp[0].cpu()).convert('RGB')
    image = Image.open(inp).convert('RGB')
    gt_mask = gt[0].squeeze().cpu().numpy()
    pred_mask = pred[0].squeeze().detach().cpu().numpy()

    # 시각화
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title('Input Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title('Ground Truth')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title('Prediction')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
    # if save_path:
    #     plt.savefig(save_path)
    #     print(f"Saved to {save_path}")
    # else:
    #     plt.show()

In [ ]:
# inference

#pbar = tqdm(loader, leave=False, desc='val')

for i, batch in enumerate(loader):
    if i >= 50:
        break

    for k, v in batch.items():
        batch[k] = v.to(device)

    inp = batch['inp']
    gt = batch['gt']

    pred = torch.sigmoid(model.infer(inp))

    inp = os.path.join(input_path, input_imgs[i])
    visualize_batch_sample(inp, gt, pred)